In [10]:
import tensorflow as tf

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\morit\Downloads\pythondev\dog_cat_photos",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\morit\Downloads\pythondev\dog_cat_photos",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)
def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label

train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)

train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)

train_dataset = train_dataset.shuffle(32)

input_layer = tf.keras.Input(shape=(224, 224, 3))   # 入力層
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)   # 前処理（正規化）をする層

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(224, 224, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False

output_layer = tf.keras.layers.Dense(1, activation='sigmoid')

# base_modelに先ほどのDense層を追加したモデルを作成する
model = tf.keras.Sequential([
    base_model,
    output_layer
])

# modelをcompileする
model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])

# modelに学習させる
model.fit(train_dataset, epochs=20)

pred_data = model.predict(test_dataset)
model.evaluate(test_dataset)

Found 400 files belonging to 2 classes.
Found 400 files belonging to 2 classes.
Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 38s 435ms/step - accuracy: 0.7103 - loss: 0.6067
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 33s 419ms/step - accuracy: 0.7817 - loss: 0.4905
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 32s 397ms/step - accuracy: 0.7950 - loss: 0.4518
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 31s 396ms/step - accuracy: 0.8020 - loss: 0.4231
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 33s 412ms/step - accuracy: 0.8227 - loss: 0.3945
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 37s 461ms/step - accuracy: 0.8205 - loss: 0.3908
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 37s 461ms/step - accuracy: 0.8470 - loss: 0.3557
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 35s 434ms/step - accuracy: 0.8514 - loss: 0.3502
Epoch 9/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 34s 428ms/step - accuracy: 0.8547 - loss: 0.3380
Epoch 10/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 33s 419ms/step - accuracy: 0.8695 - loss: 0.3271
Epoch 11/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 34s 427ms/

[0.257512629032135, 0.9100000262260437]